In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

import os, sys
os.chdir('..')

from sazz.samplers.AutomaticBoomerangSampler import AutomaticBoomerangSampler
from sazz.samplers.StickyAutomaticBoomerangSampler import StickyAutomaticBoomerangSampler
from sazz.models.bnn_torch import make_bnn_regression

In [ ]:
rng = np.random.default_rng(42)

# --- Ground truth: heteroscedastic multimodal function ---
# A smooth nonlinear trend with a region of low density (data gap),
# letting you visualise posterior uncertainty away from training data.
def true_f(x):
    return np.sin(2.0 * x) + 0.3 * np.cos(5.0 * x) + 0.5 * np.tanh(x)

# --- Training inputs with a gap in the middle ---
# This creates "in-distribution" and "out-of-distribution" test regions,
# which is the classic BNN calibration test.
n_left = 40
n_right = 40
x_left = rng.uniform(-3.0, -0.5, size=n_left)
x_right = rng.uniform(0.5, 3.0, size=n_right)
x_train = np.concatenate([x_left, x_right])
noise_std_true = 0.1
y_train = true_f(x_train) + rng.normal(0, noise_std_true, size=x_train.shape)

# --- Test inputs cover full range including the gap ---
x_test = np.linspace(-4.0, 4.0, 300)
y_test = true_f(x_test)

# --- Standardise (matching your UCI convention) ---
x_mean, x_std = x_train.mean(), x_train.std()
y_mean, y_std = y_train.mean(), y_train.std()

x_train_s = (x_train - x_mean) / x_std
x_test_s = (x_test - x_mean) / x_std
y_train_s = (y_train - y_mean) / y_std
y_test_s = (y_test - y_mean) / y_std

# --- Torch tensors ---
datasets_1d = {
    "synthetic_1d": {
        "X_train": torch.tensor(x_train_s.reshape(-1, 1), dtype=torch.float64),
        "y_train": torch.tensor(y_train_s, dtype=torch.float64),
        "X_test":  torch.tensor(x_test_s.reshape(-1, 1),  dtype=torch.float64),
        "y_test":  torch.tensor(y_test_s,  dtype=torch.float64),
        "x_mean": x_mean, "x_std": x_std,
        "y_mean": y_mean, "y_std": y_std,
        "x_train_raw": x_train, "y_train_raw": y_train,
        "x_test_raw": x_test, "y_test_raw": y_test,
        "noise_std_true": noise_std_true,
    }
}

print(f"synthetic_1d  N={len(x_train)}  D=1  "
      f"train={len(x_train)}  test={len(x_test)}")

In [ ]:
target_1d = make_bnn_regression(
    datasets_1d['synthetic_1d']['X_train'],
    datasets_1d['synthetic_1d']['y_train'],
    layer_sizes=[1, 20, 1],     # small enough to compare to HMC
    activation="tanh",
    prior_std_weight=1.0,
    prior_std_bias=1.0,
    fan_in_scaling=True,
    noise_std=0.1 / datasets_1d['synthetic_1d']['y_std'],  # match standardised noise
)

In [ ]:
from sazz.utils.bnn_utils import make_kappa_vector_bnn
from sazz.utils.warmup import warmup

sampler_boston = AutomaticBoomerangSampler(
    grad_target=target_1d.grad_target,
    D=target_1d.D,
    thinning="pli",
    refresh_rate=0.1,
)

sampler_boston.preprocess(x_ref=target_1d.x_ref, Sigma_inv=target_1d.Sigma_inv)

sampler_boston.preprocess(
    x_ref=target_1d.x_ref,
    Sigma_inv=target_1d.Sigma_inv
)
#warmup(sampler_boston, n_rounds=3, n_pilot=1000, target=target_boston)

# Kappa per layer — middle layers sparser
kappa_boston = make_kappa_vector_bnn(
    target_1d.meta['layer_sizes'],
    #kappa_weights=[0.5, 0.05, 0.5],  # sparse middle layer
    kappa_weights=[1.0, 1.0],
    kappa_biases=1e6,
)

sampler_boston_sticky = StickyAutomaticBoomerangSampler(
    grad_target=target_1d.grad_target,
    D=target_1d.D,
    thinning="pli",
    refresh_rate=0.1,
    kappa=kappa_boston
)

# Reference matches the prior — principled and needs no tuning
sampler_boston_sticky.preprocess(
    x_ref=target_1d.x_ref,
    Sigma_inv=target_1d.Sigma_inv
)

In [ ]:
# --- Sample ---
N_SKELETON=10_000
result_boston = sampler_boston.sample(N=N_SKELETON, diagnostics=True)
result_boston_sticky = sampler_boston_sticky.sample(N=N_SKELETON, diagnostics=True)

In [ ]:
import pymc as pm
import numpy as np

# --- Match the BNN priors and architecture ---
layer_sizes = [1, 20, 1]
activation_np = np.tanh
prior_std_weight = 1.0
prior_std_bias = 1.0
fan_in_scaling = True

X_train_np = datasets_1d['synthetic_1d']['X_train'].numpy()
y_train_np = datasets_1d['synthetic_1d']['y_train'].numpy()
noise_std_standardised = 0.1 / datasets_1d['synthetic_1d']['y_std']  # same as used in the Boomerang

# Per-layer prior stds (He-style if fan_in_scaling)
def prior_std(n_in):
    return prior_std_weight / np.sqrt(n_in) if fan_in_scaling else prior_std_weight

with pm.Model() as bnn_model:
    # Layer 0: input (1) -> hidden (20)
    W0 = pm.Normal('W0', mu=0.0, sigma=prior_std(layer_sizes[0]),
                   shape=(layer_sizes[1], layer_sizes[0]))
    b0 = pm.Normal('b0', mu=0.0, sigma=prior_std_bias,
                   shape=(layer_sizes[1],))

    # Layer 1: hidden (20) -> output (1)
    W1 = pm.Normal('W1', mu=0.0, sigma=prior_std(layer_sizes[1]),
                   shape=(layer_sizes[2], layer_sizes[1]))
    b1 = pm.Normal('b1', mu=0.0, sigma=prior_std_bias,
                   shape=(layer_sizes[2],))

    # Forward pass — matches your BNNLikelihood.predict
    h = pm.math.tanh(X_train_np @ W0.T + b0)
    mu = (h @ W1.T + b1).squeeze(-1)

    # Likelihood
    pm.Normal('y', mu=mu, sigma=noise_std_standardised, observed=y_train_np)

    nuts_trace = pm.sample(
        draws=2000, tune=1000, chains=2,
        target_accept=0.9, progressbar=True, random_seed=0,
    )

# --- Flatten to match the Boomerang parameterisation ---
# Convention: [W0 flat, b0, W1 flat, b1]
W0_s = nuts_trace.posterior['W0'].values.reshape(-1, layer_sizes[1] * layer_sizes[0])
b0_s = nuts_trace.posterior['b0'].values.reshape(-1, layer_sizes[1])
W1_s = nuts_trace.posterior['W1'].values.reshape(-1, layer_sizes[2] * layer_sizes[1])
b1_s = nuts_trace.posterior['b1'].values.reshape(-1, layer_sizes[2])

samples_nuts = np.concatenate([W0_s, b0_s, W1_s, b1_s], axis=1)
print(f"NUTS: {samples_nuts.shape[0]} samples × {samples_nuts.shape[1]} parameters")
assert samples_nuts.shape[1] == target_1d.D, \
    f"Shape mismatch: NUTS gave {samples_nuts.shape[1]}, target.D={target_1d.D}"

In [ ]:
# ============================================================================
# Evaluate: RMSE, log-lik
# ============================================================================
from sazz.utils.sampling import resample_pdmp_path, resample_pdmp_path_sticky
from sazz.models.bnn_torch import predict_regression

BURNIN_FRAC = 0.1
N_RESAMPLE = 50_000

def resample_and_predict(result, target, sticky=False):
    rsm = resample_pdmp_path_sticky if sticky else resample_pdmp_path
    samples_np = rsm(
        result["positions"].cpu().numpy(),
        result["velocities"].cpu().numpy(),
        result["times"].cpu().numpy(),
        target.x_ref.cpu().numpy(),
        N_resample=N_RESAMPLE,
        burnin_frac=BURNIN_FRAC,
    )
    samples = torch.tensor(samples_np, dtype=torch.float64)
    mean, std = predict_regression(samples, X_test_1d, target)
    return samples, mean, std


def metrics(mean_pred, std_pred, y_test, noise_std):
    rmse = ((mean_pred - y_test) ** 2).mean().sqrt()
    total_std = (std_pred ** 2 + noise_std ** 2).sqrt()
    log_lik = (
        -0.5 * ((y_test - mean_pred) / total_std) ** 2
        - total_std.log()
        - 0.5 * torch.log(torch.tensor(2 * torch.pi))
    ).mean()
    return rmse, log_lik


X_test_1d = datasets_1d['synthetic_1d']['X_test']
y_test_1d = datasets_1d['synthetic_1d']['y_test']
noise_std = target_1d.meta["model"].likelihood.noise_std

samples_b,  mean_b,  std_b  = resample_and_predict(result_boston, target_1d)
samples_bs, mean_bs, std_bs = resample_and_predict(result_boston_sticky, target_1d, sticky=True)

rmse_b,  ll_b  = metrics(mean_b,  std_b,  y_test_1d, noise_std)
rmse_bs, ll_bs = metrics(mean_bs, std_bs, y_test_1d, noise_std)

print(f"{'Method':<20}{'RMSE':>10}{'LogLik':>10}{'Pred std':>12}")
print("-" * 52)
print(f"{'Boomerang':<20}{rmse_b:>10.4f}{ll_b:>10.4f}{std_b.mean():>12.4f}")
print(f"{'Sticky Boomerang':<20}{rmse_bs:>10.4f}{ll_bs:>10.4f}{std_bs.mean():>12.4f}")

In [ ]:
# ============================================================================
# Plot: posterior predictive vs. true function
# ============================================================================
import matplotlib.pyplot as plt

data = datasets_1d['synthetic_1d']
x_test_raw = data['x_test_raw']
y_test_raw = data['y_test_raw']
x_train_raw = data['x_train_raw']
y_train_raw = data['y_train_raw']
y_mean_s, y_std_s = data['y_mean'], data['y_std']

def to_original_scale(mean_s, std_s):
    """Undo standardisation: y = y_s * y_std + y_mean, std likewise."""
    return mean_s * y_std_s + y_mean_s, std_s * y_std_s

# Build total predictive std (epistemic + observation noise, in original units)
noise_std_orig = noise_std * y_std_s

mean_b_orig,  std_b_orig  = to_original_scale(mean_b,  std_b)
mean_bs_orig, std_bs_orig = to_original_scale(mean_bs, std_bs)
total_b  = (std_b_orig  ** 2 + noise_std_orig ** 2).sqrt()
total_bs = (std_bs_orig ** 2 + noise_std_orig ** 2).sqrt()

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, mean, total, title in [
    (axes[0], mean_b_orig.numpy(),  total_b.numpy(),  "Boomerang"),
    (axes[1], mean_bs_orig.numpy(), total_bs.numpy(), "Sticky Boomerang"),
]:
    ax.plot(x_test_raw, y_test_raw, 'k-', lw=1.5, label='true function')
    ax.plot(x_train_raw, y_train_raw, 'k.', ms=4, alpha=0.6, label='training data')
    ax.plot(x_test_raw, mean, 'C0-', lw=2, label='posterior mean')
    ax.fill_between(x_test_raw, mean - 2 * total, mean + 2 * total,
                    color='C0', alpha=0.25, label=r'$\pm 2\sigma$')
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.legend(loc='upper left', fontsize=9)
axes[0].set_ylabel('y')
plt.tight_layout()
plt.show()